In [0]:
#Adding of Columns (Data Enrichment)
def read_csv_df(spark,path,header=True,infer_schema=True,sep=","):
    return (spark.read
        .option("header", header)
        .option("inferSchema", infer_schema)
        .option("sep", sep)
        .csv(path))

df1=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source1.txt',True,True,',')
df1.display()
df2=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source2.txt',True,True,',')
df2.display()

In [0]:
#Add Audit Timestamp
from pyspark.sql.functions import current_timestamp
df1=df1.withColumn('load_dt',current_timestamp())
df2=df2.withColumn('load_dt',current_timestamp())
df1.display()
df2.display()

In [0]:
#2. Create Full Name
from pyspark.sql.functions import concat,lit
df1=df1.withColumn('full_name',concat(df1.first_name, lit(' '), df1.last_name))
df2=df2.withColumn('full_name',concat(df2.first_name, lit(' '), df2.last_name))
df1.display()
df2.display()

In [0]:
#Define Route Segment
from pyspark.sql.functions import concat,lit
def read_json(apark,path,mline):
    df = spark.read.json(path, multiLine=mline)
    return df

jsondf=read_json(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_shipment_detail_3000.json',True)
jsondf = jsondf.withColumn('route_segment', concat(jsondf.source_city, lit('-'), jsondf.destination_city))
jsondf.display()

In [0]:
#4. Generate Vehicle Identifier
from pyspark.sql.functions import concat,lit

jsondf=jsondf.withColumn('vehicle_identifier', concat(jsondf.vehicle_type, lit('_'), jsondf.shipment_id))
jsondf.display()

In [0]:
#Deriving of Columns (Time Intelligence)
from pyspark.sql.functions import year, to_date
#1. Derive Shipment Year
jsondf = jsondf.withColumn('shipment_year', year(to_date(jsondf.shipment_date, 'yy-MM-dd')))
jsondf.display()

In [0]:
#Derive Shipment Month
from pyspark.sql.functions import to_date,month
jsondf = jsondf.withColumn('shipment_month', month(to_date(jsondf.shipment_date, 'yy-MM-dd')))
jsondf.display()

In [0]:
#Flag Weekend Operations
from pyspark.sql.functions import to_date,month,dayofweek,when
#Read JSON File
jsondf = jsondf.withColumn('is_weekend', when(dayofweek(to_date(jsondf.shipment_date, 'yy-MM-dd')).isin([1,7]), 'True').otherwise('False'))
jsondf.display()

In [0]:
#Flag shipment status
from pyspark.sql.functions import when
#Read JSON File
jsondf = jsondf.withColumn('is_expedited', when((jsondf.shipment_status).isin(['IN_TRANSIT']) | (jsondf.shipment_status).isin(['DELIVERED']), 'True').otherwise('False'))
jsondf.display()

In [0]:
#Enrichment/Business Logics (Calculated Fields)
#Calculate Unit Cost
from pyspark.sql.functions import col, try_divide, round
jsondf = jsondf.withColumn('cost_per_kg', round(try_divide(col('shipment_cost'), col('shipment_weight_kg')), 2))
jsondf.display()

In [0]:
#Track Shipment Age
from pyspark.sql.functions import datediff, to_date
jsondf = jsondf.withColumn('days_since_shipment', datediff(to_date(current_timestamp(), 'yyyy-MM-dd'), to_date(jsondf.shipment_date, 'yy-MM-dd')))
jsondf.display()

In [0]:
#Compute Tax Liability
from pyspark.sql.functions import datediff, to_date
jsondf = jsondf.withColumn('tax_amount', round((jsondf.shipment_cost * 0.18),2))
jsondf.display()

In [0]:
#Remove/Eliminate (drop, select, selectExpr)
# Remove Redundant Name Columns
df1 = df1.drop("first_name","last_name")
df2 = df2.drop("first_name","last_name")
df1.display()
df2.display()

In [0]:
#Splitting & Merging/Melting of Columns
#Splitting (Extraction) 
#Split Order Code:
from pyspark.sql.functions import substring,day,concat
jsondf = jsondf.withColumn('order_prefix', substring(jsondf.order_id, 1, 3))
jsondf = jsondf.withColumn('order_suffix', substring(jsondf.order_id, 4, 6))
#Split Date:
jsondf = jsondf.withColumn('ship_date', day(to_date(jsondf.shipment_date, 'yy-MM-dd')))
#Merging (Concatenation)
jsondf = jsondf.withColumn('route_lane ', concat(jsondf.source_city, lit('->'), jsondf.destination_city))
jsondf.display()
